Today, we are going to write the smallest, most fundamental neural network possible. We will take a single number, feed it into a single artificial neuron, measure how wrong its prediction is, and force it to learn the correct answer.

These exact 5 steps form the foundation of every single modern AI, from simple linear regression to a multi-billion-parameter LLM like GPT or Claude.

### 1. The Core Concepts (The Vocabulary)
4 fundamental terms we need to know:
1) **Tensors**: The pytorch's term for grid of numbers, A 1D tensor is a array, 2D is a matrix.
2) **Linear Layer(neuron)**: the part of the network which holds y=mx+b operation.
3) **Optimizer**: Function which lets you know the distance between the predicted and true values so that the process can adjust the values to get the distance very less.
4) **BackPropogation (backward())**: The derivative math which figures out, how much should the w and b should be updated in the formula.

- lets see a code which learns y = 2x




In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# STEP 1: Define the Data (Tensors)
# Input x = 3.0, Target y = 6.0
# PyTorch expects shape [batch_size, feature_size], so we use [[3.0]]
x = torch.tensor([[3.0]])
y_true = torch.tensor([[6.0]])

# STEP 2: Create the Neural Network
# nn.Linear(1, 1) creates 1 input feature and 1 output feature (y = wx + b)
model = nn.Linear(in_features=1, out_features=1)

# Set up Loss Function (Mean Squared Error) and Optimizer (SGD)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)  # lr = learning rate

print(f"Initial random weight (w): {model.weight.item():.4f}")
print(f"Initial random bias (b):   {model.bias.item():.4f}\n")

# STEP 3: The Training Loop (Watch it learn!)
for epoch in range(1, 101):
    # A. Forward Pass: Make a prediction
    y_pred = model(x)
    
    # B. Calculate Loss: Measure error
    loss = loss_fn(y_pred, y_true)
    
    # C. Zero Gradients: Clear out old math from previous step
    optimizer.zero_grad()
    
    # D. Backward Pass: Calculate calculus gradients (Backprop)
    loss.backward()
    
    # E. Optimizer Step: Update the weights (Learn!)
    optimizer.step()
    
    # Print progress every 20 steps
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.6f} | Prediction: {y_pred.item():.4f}")

print("\n--- Final Results ---")
print(f"Trained weight (w): {model.weight.item():.4f}  (Target is ~2.0)")
print(f"Trained bias (b):   {model.bias.item():.4f}  (Target is ~0.0)")

Initial random weight (w): 0.1393
Initial random bias (b):   -0.6046

Epoch  20 | Loss: 0.007949 | Prediction: 5.9108
Epoch  40 | Loss: 0.000001 | Prediction: 5.9990
Epoch  60 | Loss: 0.000000 | Prediction: 6.0000
Epoch  80 | Loss: 0.000000 | Prediction: 6.0000
Epoch 100 | Loss: 0.000000 | Prediction: 6.0000

--- Final Results ---
Trained weight (w): 1.9953  (Target is ~2.0)
Trained bias (b):   0.0140  (Target is ~0.0)


Let's see what happened here step by step:
1. `y_pred = model(x)`: **Forward pass**, the 3 flows into the neuron and calculates taking some random weight and bias values.
2. `loss = loss_fn(y_pred, y_true)`: **Error Measurement**, calculating the loss $(y_{\text{pred}} - y_{\text{true}})^2$. if prediction is lesser than the true value, the loss would be higher, the more we reduce the loss, the closer the prediction gets to true value, thats the point we achieve better predictions.
3. `optimizer.zero_grad()`: **Reset**, PyTorch accumulates gradients by default. We clear the memory bucket before calculating new gradients.
4. `loss.backward()`: **Backprop**, this is where the main process starts. PyTorch calculates the exact derivative of the loss with respect to weight $w$ and bias $b$. It asks: "If I tweak $w$ slightly up, does the error go up or down?"
5. `optimizer.step()`: **Weight update**, The optimizer nudges $w$ and $b$ in the direction that lowers the error:$$w_{\text{new}} = w_{\text{old}} - (\text{learning rate} \times \text{gradient})$$

> Now lets understand some things which are basics but helps to recall and understand more.

**1. related to weight update. how does it update?, how comes the process knows to increase or reduce the weights to get less loss.**
- when we call `loss.backprop()`, pytorch internally does all the calculus things with respect to weight. $$\text{Gradient} = \frac{d(\text{Loss})}{dw}$$ 
- the gradient represents the exact slope of the error curve at the current position, and based one that value being negative or positive decides to increase or decrease the weight values, and here is something about this more.
    - if the value is -ve, the weights are increased 
    - if the value is +ve, the weights are decreased
**2. where does learning rate fit in the process as we cant see it anywhere usually.**
- learning rate is represented as $\alpha$ and lives inside weight update formula. $$w_{\text{new}} = w_{\text{old}} - (\alpha \times \text{Gradient})$$
- The gradient tells the optimizer the direction and steepness of the slope, but the learning rate restricts how far it steps.
    - If LR is too low (0.00001): The subtracted value is microscopically small. It takes millions of steps to reach the bottom of the valley.
    - If LR is too high (1.0 or 10.0): The subtracted value is massive. The optimizer steps completely over the valley and lands higher up on the opposite side, causing the loss to explode to infinity (divergence).
**3. Scikit-Learn vs. Deep Learning:** 
In traditional ML (Scikit-Learn), whether gradients are used depends entirely on the algorithm:
- Linear/SGD Models: Algorithms like SGDRegressor or LogisticRegression use the exact same gradient math and learning rates. The C++ code under the hood calculates the gradient, takes the step, and zeros out the memory automatically so you never see it.

- Tree Models: Algorithms like Random Forests or XGBoost do not use gradients or learning rates in this way. They split data using statistical impurity metrics (like Gini or Entropy), requiring entirely different math.

- *In DL, no matter what we are doing there, If an architecture is part of the deep learning family and trains via PyTorch/TensorFlow, it must be end-to-end differentiable. Therefore, even if its structure mimics a tree, it will always compute gradients and rely on a learning rate.*

**4. why does pytorch keeps gradients (and why delete them)**
- pytorch forces us to call `optimizer.zero_grad()` because its behavior is to add (accumulate) new gradients to old ones, rather than overwriting them.
- This is not a memory/RAM issue; it is a mathematical feature designed for Gradient Accumulation when training massive LLMs.

- A model like GPT-4 requires batch sizes of thousands of examples to get an accurate gradient (slope) before taking a step. However, a modern GPU might only have enough RAM to process 4 examples at a time. To solve this, engineers:

1) Run 4 examples through the model.
2) Call `loss.backward()` (saves the gradients).
3) Run the next 4 examples.
4) Call `loss.backward()` (adds these new gradients to the first batch).
5) Repeat until they have processed 1,000 examples.
6) Call `optimizer.step()` to update the weights based on the combined mathematical slope of all 1,000 examples.
7) Call `optimizer.zero_grad()` to wipe the slate clean for the next batch.

If you don't zero the gradients when you are supposed to, the model mathematically combines the slope of your current step with the slope of where you were 50 steps ago, causing the optimizer to walk in random, corrupted directions.

**5. Does the model remember its previous loss so that it can know that whether the present loss decreased or increased?, so that it can know whether to increase the value of weight or reduce that** 
- Traditional human logic says: *"To know if I am getting better, I must compare my new score to my old score."* But Deep Learning does not use human logic—it uses Calculus.
- Let's break down exactly how it works on a 2D line, address how the bias gets updated, and then see a visualization of this exact process.
1. The Myth of "Memory"
- Let's use a 2D example. We want to predict $y = 2x$.
- Our true data point is $x=3$, target $y=6$.
- Let's pretend our model's current random guess is a weight ($m$) of 1.0 and a bias ($b$) of 0.0.
- So our prediction is: $y = (1.0 * 3) + 0.0 = 3$.
- Our loss is $(3 - 6)^2 = 9$.
- Here is the magic: The model does not need to remember that the loss was 9, try a weight of 1.1, get a loss of 8, and say "Aha! 1.1 is better than 1.0!"
- Instead, PyTorch applies a calculus derivative formula to the loss function at the exact moment the weight is 1.0.
- The derivative formula is: $2 \times (\text{Prediction} - \text{Target}) \times \text{Input}$.
So: $2 \times (3 - 6) \times 3 = -18$.}
- The derivative (gradient) is -18.
    - The negative sign means: "If you increase the weight, the loss will go down."
    - The large number (18) means: "The slope is very steep, you are far from the bottom."
The optimizer updates the weight by subtracting the gradient (multiplied by the learning rate). It didn't need to try two values and compare them. Calculus gave it a mathematical compass pointing exactly toward zero.

**6. Till now we got to know about weight update, but what about bias update? here is what actually happens**
- You asked why I only talked about the weight. The beautiful thing about backpropagation is that it works for every parameter simultaneously.
- When you call loss.backward(), PyTorch calculates two separate derivatives at the exact same time:
    - The derivative with respect to Weight ($m$).
    - The derivative with respect to Bias ($b$).
- The derivative for the bias tells the optimizer: "If I hold the weight completely still, and just nudge the line up or down, will the error decrease?"
- The optimizer then updates the weight and the bias at the exact same moment. The weight tilts the slope of the line, and the bias shifts the line up and down on the Y-axis.
